# Attachments EDA — Phase 1

Profiles the attachments across the historical IDMT tickets: how many per ticket, file
types, sizes, extracted characters, description length, and how many tickets exceed the
token budget when their attachment text is combined.

**Setup** (on a box with Jira + IDP creds, `.env` configured):
```
uv sync --extra eda --extra extract
```
Heavy logic lives in `scripts/eda_attachments.py`; this notebook is the interactive view.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, 'scripts')  # import the eda module
import eda_attachments as e
from IPython.display import Image, display

TICKETS = 'tickets.txt'           # one IDMT id per line
OUT = Path('out/eda/attachments')
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / 'attachments_raw.json'


## 1. Collect (live Jira) — cached
First run fetches + extracts every attachment (slow). It caches to `attachments_raw.json`;
set `USE_CACHE = True` afterwards to re-run the analysis instantly without re-fetching.


In [ ]:
USE_CACHE = CACHE.exists()
if USE_CACHE:
    records = json.loads(CACHE.read_text(encoding='utf-8'))
    print(f'loaded {len(records)} cached records')
else:
    ids = e.load_ticket_ids(TICKETS)
    print(f'collecting {len(ids)} tickets...')
    records = await e.collect(ids, concurrency=4)   # notebook top-level await
    CACHE.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'cached {len(records)} records -> {CACHE}')


In [ ]:
att, tickets = e.to_frames(records)
print('attachments:', len(att), '| tickets:', len(tickets))
att.head()


In [ ]:
tickets.head()


## 2. Compute metrics + charts
`build_charts` writes the PNGs (for the docx) and returns the metric tables. We display
each chart + its stats below.


In [ ]:
stats, charts = e.build_charts(att, tickets, OUT / 'charts')
(OUT / 'stats.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
print('sections:', list(charts))


### Attachments per ticket


In [ ]:
display(Image(charts['attachments_per_ticket'])); stats['attachments_per_ticket']


### File types (what kinds, what's most common)


In [ ]:
display(Image(charts['file_types'])); stats['file_types']


### Attachment size


In [ ]:
display(Image(charts['attachment_size_kb'])); stats['attachment_size_kb']


### Extracted characters per attachment (supported types)


In [ ]:
display(Image(charts['extracted_chars'])) if 'extracted_chars' in charts else None
stats.get('extracted_chars')


### Jira description length


In [ ]:
display(Image(charts['description_chars'])); stats['description_chars']


### Combined tokens per ticket vs the 20k budget
How many tickets exceed the token budget once description + all attachment text is combined.


In [ ]:
display(Image(charts['combined_tokens'])); stats['combined_tokens']


### Coverage (idea-card presence, supported vs unsupported, extract failures)


In [ ]:
stats['coverage']


## 3. Export to .docx
Captures the metrics + charts into a Word report.


In [ ]:
docx_path = OUT / 'attachments_eda.docx'
e.build_docx(stats, charts, docx_path, n_tickets=len(tickets))
print('wrote', docx_path)
